# ONNX Image Classifier With TVM

This notebook follows the same compile/export/run workflow as `00_tvm_prep_guide.ipynb`, but the model TVM compiles is an ONNX image classifier.

The notebook first exports pretrained torchvision ResNet18 to ONNX so the example is reproducible. After that point, the TVM path uses the ONNX frontend: load `.onnx`, import it into Relay, compile for a selected target, export graph-executor artifacts, and classify `examples/assets/cat.png` with ImageNet labels.


## What TVM Is Doing Here

The workflow has five stages:

1. Export or provide an ONNX classifier model.
2. Load the ONNX file and import it into TVM Relay with `relay.frontend.from_onnx`.
3. Compile Relay for a target profile from `compilation/targets.py`.
4. Export `model.so`, `model.json`, `model.params`, `metadata.json`, and `labels.txt`.
5. Run the exported artifacts with the Python graph executor on a real image.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

EXAMPLES = ROOT / 'examples' / 'python'
COMPILATION = ROOT / 'compilation'
for path in (EXAMPLES, COMPILATION):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print('Python:', sys.version.split()[0])
print('Repository:', ROOT)


Python: 3.11.15
Repository: /workspaces/TVM-Prep-guide


## Validate The Environment

The ONNX path needs PyTorch/TorchVision for the export step, ONNX for loading the file, and TVM for compilation/runtime validation.


In [2]:
import shutil
import subprocess

from targets import TARGETS

checks = {
    'TVM': 'import tvm; print(tvm.__version__)',
    'PyTorch': 'import torch; print(torch.__version__)',
    'TorchVision': 'import torchvision; print(torchvision.__version__)',
    'ONNX': 'import onnx; print(onnx.__version__)',
}

for label, snippet in checks.items():
    result = subprocess.run([sys.executable, '-c', snippet], text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f'{label} import failed:\n{result.stderr}')
    print(f'{label}:', result.stdout.strip().splitlines()[-1])

print('\nToolchain:')
for tool in ['gcc', 'g++', 'aarch64-linux-gnu-gcc', 'arm-linux-gnueabihf-gcc', 'cmake', 'ninja']:
    print(f'{tool}:', shutil.which(tool))


TVM: 0.19.0
PyTorch: 2.3.1+cu121
TorchVision: 0.18.1+cu121
ONNX: 1.16.2

Toolchain:
gcc: /usr/bin/gcc
g++: /usr/bin/g++
aarch64-linux-gnu-gcc: /usr/bin/aarch64-linux-gnu-gcc
arm-linux-gnueabihf-gcc: /usr/bin/arm-linux-gnueabihf-gcc
cmake: /usr/bin/cmake
ninja: /usr/bin/ninja


## Pick A Target Profile

Use `x86_64` for local notebook validation inside the devcontainer. Change `TARGET_PROFILE` to a cross target, such as `raspi4_aarch64`, after the local ONNX path works.


In [3]:
for name, profile in sorted(TARGETS.items()):
    print(name)
    print(f'  target: {profile["target"]}')
    if 'host' in profile:
        print(f'  host:   {profile["host"]}')
    if 'cc' in profile:
        print(f'  cc:     {profile["cc"]}')
    print(f'  output: model.{profile["ext"]}')
    print()

TARGET_PROFILE = 'x86_64'


c
  target: c
  output: model.tar

native
  target: llvm -mcpu=native
  cc:     gcc
  output: model.so

raspi4_aarch64
  target: llvm -mtriple=aarch64-linux-gnu -mcpu=cortex-a72 -mattr=+neon
  host:   llvm -mtriple=aarch64-linux-gnu
  cc:     aarch64-linux-gnu-gcc
  output: model.so

raspi_armv7
  target: llvm -mtriple=armv7-linux-gnueabihf -mcpu=cortex-a72 -mattr=+neon,+vfp3
  host:   llvm -mtriple=armv7-linux-gnueabihf
  cc:     arm-linux-gnueabihf-gcc
  output: model.so

vulkan
  target: vulkan
  host:   llvm -mtriple=x86_64-linux-gnu
  output: model.so

wasm32
  target: llvm -mtriple=wasm32-unknown-unknown-wasm
  cc:     emcc
  output: model.wasm

x86_64
  target: llvm -mtriple=x86_64-linux-gnu -mcpu=x86-64
  cc:     gcc
  output: model.so



## Export A Pretrained Classifier To ONNX

This cell creates an ONNX file from torchvision ResNet18. The pretrained weights provide ImageNet labels, and those labels are exported with the TVM artifacts later.

If you already have an ONNX model, replace `ONNX_PATH`, `INPUT_NAME`, and `INPUT_SHAPE` with your model's values and skip the export call.


In [4]:
import torch
import torchvision.models as models
from torchvision.models import ResNet18_Weights

MODEL_NAME = 'resnet18_onnx'
INPUT_NAME = 'input0'
INPUT_SHAPE = (1, 3, 224, 224)
ONNX_PATH = ROOT / 'examples' / 'artifacts' / 'onnx_models' / 'resnet18.onnx'
LABELS_PATH = ROOT / 'examples' / 'artifacts' / 'onnx_models' / 'imagenet_labels.txt'
ONNX_PATH.parent.mkdir(parents=True, exist_ok=True)

weights = ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights).eval()
labels = list(weights.meta['categories'])
example = torch.zeros(INPUT_SHAPE, dtype=torch.float32)

torch.set_grad_enabled(False)
torch.onnx.export(
    model,
    example,
    ONNX_PATH,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=[INPUT_NAME],
    output_names=['logits'],
    dynamic_axes=None,
)

LABELS_PATH.write_text('\n'.join(labels) + '\n', encoding='utf-8')

print('ONNX model:', ONNX_PATH.relative_to(ROOT))
print('Label file:', LABELS_PATH.relative_to(ROOT))
print('Input     :', INPUT_NAME, INPUT_SHAPE)
print('Labels    :', len(labels))


/home/vscode/.local/lib/python3.11/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.Op.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/home/vscode/.local/lib/python3.11/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.OnnxFunction.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()


ONNX model: examples/artifacts/onnx_models/resnet18.onnx
Label file: examples/artifacts/onnx_models/imagenet_labels.txt
Input     : input0 (1, 3, 224, 224)
Labels    : 1000


## Inspect The ONNX Model

Before compiling an ONNX model, confirm the graph input names and shapes. These must match what you pass to TVM and what the runtime later feeds into the graph executor.


In [5]:
import onnx

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

for value in onnx_model.graph.input:
    dims = []
    for dim in value.type.tensor_type.shape.dim:
        dims.append(dim.dim_value or dim.dim_param or '?')
    print(f'input:  name={value.name} shape={dims}')

for value in onnx_model.graph.output:
    dims = []
    for dim in value.type.tensor_type.shape.dim:
        dims.append(dim.dim_value or dim.dim_param or '?')
    print(f'output: name={value.name} shape={dims}')


input:  name=input0 shape=[1, 3, 224, 224]
output: name=logits shape=[1, 1000]


## Import ONNX Into Relay

`import_onnx` in `compilation/compile.py` loads the ONNX file and calls `relay.frontend.from_onnx` with the input name and shape. At this stage TVM has a Relay module and parameter dictionary, but it has not generated target-specific machine code yet.


In [6]:
from compile import import_onnx

mod, params, _ = import_onnx(str(ONNX_PATH), INPUT_NAME, INPUT_SHAPE)
print(type(mod).__name__)
print('Parameter tensors:', len(params))


IRModule
Parameter tensors: 0


## Compile And Export Artifacts

`build_and_save` compiles the Relay module for `TARGET_PROFILE` and writes the graph-executor artifact directory under `examples/artifacts/<model>/<target>/`.


In [7]:
import json
from compile import build_and_save

artifact_dir = build_and_save(
    mod,
    params,
    model_name=MODEL_NAME,
    target_name=TARGET_PROFILE,
    output_dir=ROOT / 'examples' / 'artifacts',
    labels=labels,
    input_name=INPUT_NAME,
    input_shape=INPUT_SHAPE,
)

print('Artifacts:', artifact_dir.relative_to(ROOT))
for path in sorted(artifact_dir.iterdir()):
    print('-', path.name)

metadata = json.loads((artifact_dir / 'metadata.json').read_text(encoding='utf-8'))
metadata


One or more operators have not been tuned. Please tune your model for better performance. Use DEBUG logging level to see more details.


Artifacts: examples/artifacts/resnet18_onnx/x86_64
- labels.txt
- metadata.json
- model.json
- model.params
- model.so


{'model': 'resnet18_onnx',
 'input_name': 'input0',
 'input_shape': [1, 3, 224, 224],
 'input_dtype': 'float32',
 'target': 'x86_64',
 'target_str': 'llvm -mtriple=x86_64-linux-gnu -mcpu=x86-64',
 'library': 'model.so',
 'labels': 'labels.txt'}

## Run The Compiled ONNX Model On `cat.png`

The runtime path is the same as the PyTorch notebook: preprocess the image, load the compiled graph artifacts, run the TVM graph executor, and map the top outputs to ImageNet labels.


In [8]:
from tvm_prep.preprocess import load_imagenet_image
from tvm_prep.runtime import run_graph_executor, topk

image_path = ROOT / 'examples' / 'assets' / 'cat.png'
compiled_labels = (artifact_dir / metadata['labels']).read_text(encoding='utf-8').splitlines()
image = load_imagenet_image(image_path, metadata['input_shape'], layout='NCHW')
output = run_graph_executor(artifact_dir, image)
predictions = topk(output, compiled_labels, k=5)

print('Image:', image_path.relative_to(ROOT))
print('Output shape:', output.shape)
print()
for rank, (idx, prob, label) in enumerate(predictions, start=1):
    print(f'{rank}: id={idx} p={prob:.6f} {label}')

top_id, top_prob, top_label = predictions[0]
print(f'\nPrediction: {top_label} ({top_prob:.2%})')


Image: examples/assets/cat.png
Output shape: (1, 1000)

1: id=282 p=0.921829 tiger cat
2: id=281 p=0.055034 tabby
3: id=285 p=0.011087 Egyptian cat
4: id=287 p=0.007073 lynx
5: id=539 p=0.001127 doormat

Prediction: tiger cat (92.18%)


## Command-Line Equivalent

The notebook makes each step visible. Once you already have an ONNX file, the same compile path is available through the CLI. The `--labels` file is optional for generic ONNX models, but it gives readable ImageNet output for this classifier.

```bash
python compilation/compile.py \
  --frontend onnx \
  --model-path examples/artifacts/onnx_models/resnet18.onnx \
  --input-name input0 \
  --input-shape 1,3,224,224 \
  --labels examples/artifacts/onnx_models/imagenet_labels.txt \
  --target x86_64

python examples/python/run_model.py \
  --artifact-dir examples/artifacts/resnet18/x86_64 \
  --image examples/assets/cat.png
```

For deployment, compile with a target such as `raspi4_aarch64`, then build and copy the matching runtime package:

```bash
python compilation/compile.py \
  --frontend onnx \
  --model-path examples/artifacts/onnx_models/resnet18.onnx \
  --input-name input0 \
  --input-shape 1,3,224,224 \
  --labels examples/artifacts/onnx_models/imagenet_labels.txt \
  --target raspi4_aarch64

bash compilation/build_runtime.sh raspi4_aarch64
```

Copy both `examples/artifacts/<model>/<target>/` and `examples/artifacts/runtime/<target>/` to the target device.
